# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

In [1]:
import pandas as pd

# Load the customer dataset
url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
df = pd.read_csv(url)

# Clean the column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

# Keep customers with claims below 1000 who responded Yes
low_claim_responders = df[
    (df["total_claim_amount"] < 1000) &
    (df["response"].str.lower() == "yes")
]

# Show the result
print("Number of customers:", len(low_claim_responders))
low_claim_responders.head()

Number of customers: 1399


,unnamed:_0,customer,state,customer_lifetime_value,response,coverage,education,effective_to_date,employmentstatus,gender,...,number_of_open_complaints,number_of_policies,policy_type,policy,renew_offer_type,sales_channel,total_claim_amount,vehicle_class,vehicle_size,vehicle_type
3,3,XL78013,Oregon,22332.439460,Yes,Extended,College,1/11/11,Employed,M,...,0.0,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A
8,8,FM55990,California,5989.773931,Yes,Premium,College,1/19/11,Employed,M,...,0.0,1,Personal Auto,Personal L1,Offer2,Branch,739.200000,Sports Car,Medsize,NaN
15,15,CW49887,California,4626.801093,Yes,Basic,Master,1/16/11,Employed,F,...,0.0,1,Special Auto,Special L1,Offer2,Branch,547.200000,SUV,Medsize,NaN
19,19,NJ54277,California,3746.751625,Yes,Extended,College,2/26/11,Employed,F,...,1.0,1,Personal Auto,Personal L2,Offer2,Call Center,19.575683,Two-Door Car,Large,A
27,27,MQ68407,Oregon,4376.363592,Yes,Premium,Bachelor,2/28/11,Employed,F,...,0.0,1,Personal Auto,Personal L3,Offer2,Agent,60.036683,Four-Door Car,Medsize,NaN


2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

In [2]:
# Keep only customers who responded "Yes"
responders = df[df["response"].str.lower() == "yes"]

# Compare average premium, lifetime value and claims by policy type and gender
segment_analysis = (
    responders
    .groupby(["policy_type", "gender"])
    .agg(
        average_monthly_premium=("monthly_premium_auto", "mean"),
        average_customer_lifetime_value=("customer_lifetime_value", "mean"),
        average_total_claim_amount=("total_claim_amount", "mean")
    )
    .round(2)
)

segment_analysis

average_monthly_premium  \
policy_type    gender                            
Corporate Auto F                         94.30   
               M                         92.19   
Personal Auto  F                         99.00   
               M                         91.09   
Special Auto   F                         92.31   
               M                         86.34   

                       average_customer_lifetime_value  \
policy_type    gender                                    
Corporate Auto F                               7712.63   
               M                               7944.47   
Personal Auto  F                               8339.79   
               M                               7448.38   
Special Auto   F                               7691.58   
               M                               8247.09   

                       average_total_claim_amount  
policy_type    gender                              
Corporate Auto F                           433.74  
               M                           408.58  
Personal Auto  F                           452.97  
               M                           457.01  
Special Auto   F                           453.28  
               M                           429.53

In [5]:
# Higher CLV combined with lower claims suggests a more profitable, lower-risk segment
segment_analysis.sort_values(
    by=["average_customer_lifetime_value", "average_total_claim_amount"],
    ascending=[False, True]
)

average_monthly_premium  \
policy_type    gender                            
Personal Auto  F                         99.00   
Special Auto   M                         86.34   
Corporate Auto M                         92.19   
               F                         94.30   
Special Auto   F                         92.31   
Personal Auto  M                         91.09   

                       average_customer_lifetime_value  \
policy_type    gender                                    
Personal Auto  F                               8339.79   
Special Auto   M                               8247.09   
Corporate Auto M                               7944.47   
               F                               7712.63   
Special Auto   F                               7691.58   
Personal Auto  M                               7448.38   

                       average_total_claim_amount  
policy_type    gender                              
Personal Auto  F                           452.97  
Special Auto   M                           429.53  
Corporate Auto M                           408.58  
               F                           433.74  
Special Auto   F                           453.28  
Personal Auto  M                           457.01

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

In [6]:
# Count customers per state
customers_by_state = (
    df.groupby("state")
    .agg(total_customers=("customer", "count"))
    .sort_values("total_customers", ascending=False)
)

# Keep only states with more than 500 customers
states_over_500 = customers_by_state[
    customers_by_state["total_customers"] > 500
]

states_over_500

,total_customers
state,
California,3552
Oregon,2909
Arizona,1937
Nevada,993
Washington,888


4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

In [7]:
# Calculate maximum, minimum and median CLV by education and gender
clv_summary = (
    df.groupby(["education", "gender"])
    .agg(
        maximum_clv=("customer_lifetime_value", "max"),
        minimum_clv=("customer_lifetime_value", "min"),
        median_clv=("customer_lifetime_value", "median")
    )
    .round(2)
    .sort_values("median_clv", ascending=False)
)

clv_summary

maximum_clv  minimum_clv  median_clv
education            gender                                      
High School or Below M          83325.38      1940.98     6286.73
                     F          55277.45      2144.92     6039.55
College              M          61134.68      1918.12     6005.85
Master               F          51016.07      2417.78     5729.86
Bachelor             F          73225.96      1904.00     5640.51
College              F          61850.19      1898.68     5623.61
Master               M          50568.26      2272.31     5579.10
Doctor               M          32677.34      2267.60     5577.67
Bachelor             M          67907.27      1898.01     5548.03
Doctor               F          44856.11      2395.57     5332.46

In [8]:
# Show the groups with the highest and lowest median CLV
print("Highest median CLV:")
display(clv_summary.head(1))

print("Lowest median CLV:")
display(clv_summary.tail(1))

Highest median CLV:


,,maximum_clv,minimum_clv,median_clv
education,gender,,,
High School or Below,M,83325.38,1940.98,6286.73


Lowest median CLV:


,,maximum_clv,minimum_clv,median_clv
education,gender,,,
Doctor,F,44856.11,2395.57,5332.46


## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

In [ ]:
# your code goes here